# Intergenerational Mobility Fund - Interactive Analysis

This notebook demonstrates key analytical methods using Python for the Intergenerational Mobility Fund proposal.

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. ROI Calculator

In [ ]:
def calculate_roi(annual_earnings_gain, working_years, discount_rate, program_cost):
    """Calculate ROI metrics"""
    years = np.arange(1, working_years + 1)
    discount_factors = 1 / (1 + discount_rate) ** years
    pv_earnings = annual_earnings_gain * discount_factors.sum()
    npv = pv_earnings - program_cost
    roi_ratio = pv_earnings / program_cost
    
    return {
        'pv_lifetime_gain': pv_earnings,
        'npv': npv,
        'roi_ratio': roi_ratio
    }

# Base case calculation
base_roi = calculate_roi(
    annual_earnings_gain=7800,
    working_years=35,
    discount_rate=0.03,
    program_cost=14850
)

print("ROI Calculation - Base Case:")
print(f"  PV Lifetime Gain: ${base_roi['pv_lifetime_gain']:,.2f}")
print(f"  NPV: ${base_roi['npv']:,.2f}")
print(f"  ROI Ratio: {base_roi['roi_ratio']:.2f}:1")

## 2. Sensitivity Analysis

In [ ]:
def sensitivity_analysis(base_cost, base_earnings, cost_variations, earnings_variations):
    """Two-way sensitivity analysis for ROI"""
    results = pd.DataFrame(index=cost_variations, columns=earnings_variations)
    
    for cost_var in cost_variations:
        for earnings_var in earnings_variations:
            adjusted_cost = base_cost * (1 + cost_var)
            adjusted_earnings = base_earnings * (1 + earnings_var)
            roi = calculate_roi(adjusted_earnings, 35, 0.03, adjusted_cost)['roi_ratio']
            results.loc[cost_var, earnings_var] = roi
    
    return results

# Run sensitivity analysis
cost_vars = [-0.20, 0, 0.20]
earnings_vars = [-0.20, 0, 0.20]
sens_results = sensitivity_analysis(14850, 7800, cost_vars, earnings_variations=earnings_vars)

# Display as formatted table
sens_results_formatted = sens_results.applymap(lambda x: f"{x:.1f}:1")
sens_results_formatted.index = [f"{v:+.0%}" for v in cost_vars]
sens_results_formatted.columns = [f"{v:+.0%}" for v in earnings_vars]
sens_results_formatted.index.name = "Cost Variation"
sens_results_formatted.columns.name = "Earnings Variation"

print("Sensitivity Analysis (ROI Ratios):")
display(sens_results_formatted)

## 3. Monte Carlo Simulation

In [ ]:
def monte_carlo_simulation(base_cost, base_earnings, n_simulations=10000):
    """Monte Carlo simulation for ROI uncertainty analysis"""
    np.random.seed(42)
    
    # Triangular distributions
    cost_samples = np.random.triangular(
        base_cost * 0.8, base_cost, base_cost * 1.2, n_simulations
    )
    earnings_samples = np.random.triangular(
        base_earnings * 0.8, base_earnings, base_earnings * 1.2, n_simulations
    )
    
    roi_samples = []
    for cost, earnings in zip(cost_samples, earnings_samples):
        roi = calculate_roi(earnings, 35, 0.03, cost)['roi_ratio']
        roi_samples.append(roi)
    
    roi_samples = np.array(roi_samples)
    
    return {
        'mean': np.mean(roi_samples),
        'median': np.median(roi_samples),
        'std': np.std(roi_samples),
        'percentile_10': np.percentile(roi_samples, 10),
        'percentile_90': np.percentile(roi_samples, 90),
        'samples': roi_samples
    }

# Run Monte Carlo simulation
mc_results = monte_carlo_simulation(14850, 7800, n_simulations=10000)

print("Monte Carlo Simulation Results (n=10,000):")
print(f"  Mean ROI: {mc_results['mean']:.2f}:1")
print(f"  Median ROI: {mc_results['median']:.2f}:1")
print(f"  Std Dev: {mc_results['std']:.2f}")
print(f"  10th Percentile: {mc_results['percentile_10']:.2f}:1")
print(f"  90th Percentile: {mc_results['percentile_90']:.2f}:1")

## 4. Visualize Monte Carlo Results

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(mc_results['samples'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(mc_results['mean'], color='red', linestyle='--', linewidth=2, label=f"Mean: {mc_results['mean']:.2f}:1")
plt.axvline(mc_results['median'], color='blue', linestyle='--', linewidth=2, label=f"Median: {mc_results['median']:.2f}:1")
plt.xlabel('ROI Ratio', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Monte Carlo ROI Distribution (10,000 Simulations)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Wage Progression by Sector

In [ ]:
# Sample wage progression data
sectors = ['Healthcare', 'IT/Cybersecurity', 'Manufacturing', 'Construction']
years = [0, 1, 3, 5, 10]

wage_data = {
    'Healthcare': [18.50, 21.50, 24.80, 31.20, 42.00],
    'IT/Cybersecurity': [22.00, 28.00, 35.50, 48.00, 65.00],
    'Manufacturing': [19.00, 23.00, 27.50, 34.00, 45.00],
    'Construction': [21.00, 25.50, 30.50, 38.00, 52.00]
}

plt.figure(figsize=(12, 6))
for sector in sectors:
    plt.plot(years, wage_data[sector], marker='o', linewidth=2, markersize=8, label=sector)

plt.xlabel('Years in Career', fontsize=12)
plt.ylabel('Hourly Wage ($)', fontsize=12)
plt.title('Wage Progression by Target Sector', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. SROI Calculation

In [ ]:
# SROI components
sroi_components = {
    'Reduced public assistance': 2400,
    'Reduced criminal justice': 1800,
    'Increased income tax': 1950,
    'Increased payroll tax': 1100,
    'Reduced healthcare costs': 3200,
    'Reduced elder healthcare': 2100
}

total_annual_savings = sum(sroi_components.values())
program_cost = 14850
sroi_ratio = total_annual_savings / program_cost

print("SROI Analysis:")
print(f"\nAnnual Public Value per Participant:")
for component, value in sroi_components.items():
    print(f"  {component}: ${value:,.2f}")
print(f"\n  Total Annual Savings: ${total_annual_savings:,.2f}")
print(f"  Program Cost: ${program_cost:,.2f}")
print(f"\n  SROI Ratio: {sroi_ratio:.2f}:1")

## 7. GDP Impact Projection

In [ ]:
def gdp_projection(graduates, avg_earnings_gain, multiplier, years=20):
    """Calculate GDP impact over time"""
    years_array = np.arange(1, years + 1)
    annual_earnings = graduates * avg_earnings_gain
    gdp_contribution = annual_earnings * multiplier
    cumulative_gdp = np.cumsum([gdp_contribution] * years)
    
    return pd.DataFrame({
        'Year': years_array,
        'Annual_GDP_M': [gdp_contribution / 1_000_000] * years,
        'Cumulative_GDP_M': cumulative_gdp / 1_000_000
    })

# Project GDP impact
gdp_df = gdp_projection(
    graduates=38000,
    avg_earnings_gain=7800,
    multiplier=1.4,
    years=20
)

# Display key years
key_years = [4, 9, 19]  # Years 5, 10, 20 (0-indexed)
print("GDP Impact Projection:")
print(gdp_df.iloc[key_years].to_string(index=False))

---

**Note:** This notebook provides interactive examples of the analytical methods described in the Intergenerational Mobility Fund proposal. All calculations use Python (pandas, statsmodels, scikit-learn) as the primary analytics stack.

## 8. Figure 2: Older Adults (65+) - Mentor Role Seeking vs. Current Engagement

In [ ]:
# Figure 2: Mentor Role Seeking vs. Current Engagement
regions = ['Massachusetts', 'Washington DC', 'Virginia']
seeking_pct = [42.1, 38.7, 44.3]
serving_pct = [18.2, 16.5, 19.8]

plt.figure(figsize=(12, 6))
x = np.arange(len(regions))
width = 0.35

bars1 = plt.bar(x - width/2, seeking_pct, width, label='Seeking Mentor Role', color='#2c5282')
bars2 = plt.bar(x + width/2, serving_pct, width, label='Currently Serving', color='#4299e1')

plt.xlabel('Region', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Figure 2: Older Adults (65+) - Mentor Role Seeking vs. Current Engagement', fontsize=14, fontweight='bold')
plt.xticks(x, regions)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    plt.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height), 
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    plt.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height), 
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Figure 5: Mentor Match Duration and Graduation Probability

In [ ]:
# Figure 5: Mentor Match Duration and Graduation Probability
duration_months = [0, 3, 6, 9, 12, 18, 24]
graduation_prob = [45, 58, 68, 75, 80, 85, 88]

plt.figure(figsize=(12, 6))
plt.plot(duration_months, graduation_prob, marker='o', linewidth=3, markersize=8, 
         color='#2b6cb0', label='Graduation Probability')
plt.fill_between(duration_months, graduation_prob, alpha=0.3, color='#4299e1')

plt.xlabel('Mentor Match Duration (months)', fontsize=12)
plt.ylabel('Graduation Probability (%)', fontsize=12)
plt.title('Figure 5: Mentor Match Duration and Graduation Probability', fontsize=14, fontweight='bold')
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend()

# Add annotations for key points
for x, y in zip(duration_months, graduation_prob):
    plt.annotate(f'{y:.0f}%', xy=(x, y), xytext=(0, 10), textcoords="offset points", 
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Sophisticated Multi-Panel Dashboard

In [ ]:
# Sophisticated Multi-Panel Dashboard
regions = ['Massachusetts', 'Washington DC', 'Virginia']
youth_unemployment = [16.4, 19.9, 17.5]  # Youth unemployment/underemployment rates
mentor_supply = [42.1, 38.7, 44.3]  # Mentor supply percentages
avg_wages = [
    [18.50, 21.50, 24.80, 31.20, 42.00],  # Healthcare
    [22.00, 28.00, 35.50, 48.00, 65.00],  # IT/Cybersecurity
    [19.00, 23.00, 27.50, 34.00, 45.00],  # Manufacturing
    [21.00, 25.50, 30.50, 38.00, 52.00]   # Construction
]
roi_scenarios = {'conservative': 6.2, 'base': 11.4, 'optimistic': 18.3}

# Import the dashboard function
from analysis import create_sophisticated_dashboard

# Create the dashboard
dashboard = create_sophisticated_dashboard(
    regions=regions,
    youth_unemployment=youth_unemployment,
    mentor_supply=mentor_supply,
    avg_wages=avg_wages,
    roi_scenarios=roi_scenarios
)

plt.show()

# Save the dashboard
dashboard.savefig('comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
print("Dashboard saved as 'comprehensive_dashboard.png'")

## 11. Additional Advanced Visualizations

In [ ]:
# Regional Bubble Chart
regions = ['Massachusetts', 'Washington DC', 'Virginia']
youth_count = [303100, 231500, 289600]
mentor_count = [307100, 113700, 293500]
avg_wages = [22.5, 25.0, 23.0]

from analysis import plot_regional_bubble_chart
bubble_chart = plot_regional_bubble_chart(regions, youth_count, mentor_count, avg_wages)
plt.show()

In [ ]:
# Violin Plot for Distribution Analysis
np.random.seed(42)
data_violin = {
    'Healthcare': np.random.normal(25000, 5000, 100),
    'IT/Cybersecurity': np.random.normal(32000, 8000, 100),
    'Manufacturing': np.random.normal(28000, 6000, 100),
    'Construction': np.random.normal(30000, 7000, 100)
}

from analysis import plot_violin_distributions
violin_plot = plot_violin_distributions(data_violin)
plt.show()

In [ ]:
# Time Series with Trend Lines
years = np.array([2024, 2025, 2026, 2027, 2028, 2029, 2030])
data_series = {
    'Graduates': [500, 1500, 3000, 5000, 8000, 10000, 10000],
    'ROI Ratio': [6.2, 8.5, 10.2, 11.4, 12.0, 12.5, 13.0],
    'SROI Ratio': [3.8, 4.0, 4.1, 4.2, 4.3, 4.4, 4.5]
}

from analysis import plot_time_series_with_trend
time_series = plot_time_series_with_trend(years, data_series, trend_line=True)
plt.show()

In [ ]:
# Box Plot with Strip Overlay
np.random.seed(42)
data_box = {
    'Black Youth': np.random.normal(42000, 8000, 50),
    'Hispanic Youth': np.random.normal(40000, 7500, 50),
    'White Youth': np.random.normal(38000, 7000, 50),
    'Asian Youth': np.random.normal(45000, 9000, 50)
}

from analysis import plot_box_plot_with_strip
box_plot = plot_box_plot_with_strip(data_box)
plt.show()